# Adding Additional Context to the LLM
In this notebook, we will see how we can improve the agent accuracy by providing additional information about the table fields.

<figure>
 <img src="../assets/chapter_2.png" width="60%" align="center"/></a>
<figcaption> Prompt Template Architecture </figcaption>
</figure>

<br>
<br />

## Setting the Database Connection

The below code enables us to connect to Postgres (or DuckDB) using the `get_ibis_connection` function:

In [ ]:
import sys
import os

tbl_name = "air_traffic"

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection


Setting a connection to the Postgres database:

In [ ]:
postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)


Or, setting a connection to the DuckDB database:

In [ ]:
# csv_path = project_root + "/data/air_traffic_gold.csv"
# con = get_ibis_connection(
#     backend="duckdb",
#     duckdb_csv_path=csv_path,
# )

Next, we will extract the table attributes using the `get_tbl_attr` function:

In [ ]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con=con, tbl_name=tbl_name)

schema = tbl_attr.schema

## Setting the LLM Client

We will use the `ChatOpenAI` to set the LLM client connection:

In [ ]:
from langchain_openai import ChatOpenAI

base_url = "https://api.openai.com/v1"
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o"

llm = ChatOpenAI(
  base_url=base_url, 
  api_key=api_key, 
  temperature=0, 
  model=model
  )

## Adding Additional Context

We will add to the baseline system prompt a new argument named as `additional_context`:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format. 
Please ensure that the field names in the query are enclosed in double quotes.

{additional_context}

CREATE TABLE {tbl_name} ({schema})

""".strip()

user_template = "Write a SQL query that returns: {question}"

messages = [("system", system_template), ("user", user_template)]

prompt_template = ChatPromptTemplate.from_messages(messages)


In [ ]:
chain = prompt_template | llm

In [ ]:
def basic_sql_agent(chain, question, tbl_name, schema, con, additional_context=""):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
            "additional_context": additional_context,
        }
    )
    query = llm_output.content
    print("The return SQL query:")
    print("_" * 60)
    print(query)
    print("_" * 60)
    output = con.sql(query).execute()
    return output


In [ ]:
basic_sql_agent(
    chain,
    question="How many passengers were in transit in 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context="",
    con=con,
)


In [ ]:
con.sql('SELECT DISTINCT "Activity Type Code" FROM air_traffic').execute()

In [ ]:
basic_sql_agent(
    chain,
    question="How many passengers were in transit in 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context="The 'Activity Type Code' unique values are: 'Deplaned', 'Enplaned', and 'Thru / Transit'",
    con=con,
)


In [ ]:
from sql_ai_agent.db_handler import get_character_distinct_values
from sql_ai_agent.prompt_handler import format_distinct_values_for_prompt

distinct_values = get_character_distinct_values(
    con=con, tbl_schema=tbl_attr, tbl_name=tbl_name
)

print(distinct_values)


In [ ]:
distinct_values_formatted = format_distinct_values_for_prompt(distinct_values)
print(distinct_values_formatted)


In [ ]:
basic_sql_agent(
    chain,
    question="How many passengers were in transit in 2024?",
    tbl_name=tbl_name,
    schema=schema,
    additional_context=distinct_values_formatted,
    con=con,
)
